<a href="https://colab.research.google.com/github/Gayathri2526/AIML-learning-Projects/blob/main/Diabetes-Prediction/Day4_Tune_Ensemble_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
# Check dataset size
print("Dataset shape:", df.shape)

# Check column information
df.info()

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check target distribution
print("\nOutcome distribution:")
print(df["Outcome"].value_counts())

Dataset shape: (768, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Missing values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFun

In [3]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

X_train shape: (614, 8)
X_test shape: (154, 8)

Training target distribution:
Outcome
0    400
1    214
Name: count, dtype: int64

Testing target distribution:
Outcome
0    100
1     54
Name: count, dtype: int64


In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# Create baseline Decision Tree
baseline_model = DecisionTreeClassifier(random_state=42)

# Perform 5-fold cross validation on training data
cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

# Train baseline model on complete training data
baseline_model.fit(X_train, y_train)

# Predict test data
y_pred_baseline = baseline_model.predict(X_test)

# Test accuracy
baseline_test_accuracy = accuracy_score(y_test, y_pred_baseline)

print("5-Fold CV Scores:", cv_scores)
print("Mean CV Accuracy:", round(cv_scores.mean(), 4))
print("Test Accuracy:", round(baseline_test_accuracy, 4))

5-Fold CV Scores: [0.6097561  0.67479675 0.68292683 0.67479675 0.7295082 ]
Mean CV Accuracy: 0.6744
Test Accuracy: 0.7273


In [5]:
from sklearn.model_selection import GridSearchCV

# Parameters to test
param_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train GridSearchCV
grid_search.fit(X_train, y_train)

# Best model
grid_best_model = grid_search.best_estimator_

# Test prediction
y_pred_grid = grid_best_model.predict(X_test)

# Test accuracy
grid_test_accuracy = accuracy_score(y_test, y_pred_grid)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", round(grid_search.best_score_, 4))
print("Test Accuracy:", round(grid_test_accuracy, 4))

Best Parameters: {'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Accuracy: 0.746
Test Accuracy: 0.6948


In [6]:
from sklearn.model_selection import RandomizedSearchCV

# Parameter options
param_dist = {
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 4, 6, 8],
    "criterion": ["gini", "entropy"]
}

# Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

# Train RandomizedSearchCV
random_search.fit(X_train, y_train)

# Best model
random_best_model = random_search.best_estimator_

# Test prediction
y_pred_random = random_best_model.predict(X_test)

# Test accuracy
random_test_accuracy = accuracy_score(y_test, y_pred_random)

print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", round(random_search.best_score_, 4))
print("Test Accuracy:", round(random_test_accuracy, 4))

Best Parameters: {'min_samples_split': 2, 'min_samples_leaf': 6, 'max_depth': 3, 'criterion': 'entropy'}
Best CV Accuracy: 0.7525
Test Accuracy: 0.6948


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# 5-fold cross validation
rf_cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

# Train Random Forest
rf_model.fit(X_train, y_train)

# Predict test data
y_pred_rf = rf_model.predict(X_test)

# Test accuracy
rf_test_accuracy = accuracy_score(y_test, y_pred_rf)

print("5-Fold CV Scores:", rf_cv_scores)
print("Mean CV Accuracy:", round(rf_cv_scores.mean(), 4))
print("Test Accuracy:", round(rf_test_accuracy, 4))

5-Fold CV Scores: [0.73170732 0.81300813 0.72357724 0.77235772 0.78688525]
Mean CV Accuracy: 0.7655
Test Accuracy: 0.7597


In [8]:
results = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "GridSearchCV Decision Tree",
        "RandomizedSearchCV Decision Tree",
        "Random Forest"
    ],

    "CV Accuracy": [
        cv_scores.mean(),
        grid_search.best_score_,
        random_search.best_score_,
        rf_cv_scores.mean()
    ],

    "Test Accuracy": [
        baseline_test_accuracy,
        grid_test_accuracy,
        random_test_accuracy,
        rf_test_accuracy
    ],

    "Best Params": [
        "Default Parameters",
        str(grid_search.best_params_),
        str(random_search.best_params_),
        "n_estimators=100, random_state=42"
    ]
})

results["CV Accuracy"] = results["CV Accuracy"].round(4)
results["Test Accuracy"] = results["Test Accuracy"].round(4)

results

,Model,CV Accuracy,Test Accuracy,Best Params
0,Baseline Decision Tree,0.6744,0.7273,Default Parameters
1,GridSearchCV Decision Tree,0.7460,0.6948,"{'criterion': 'entropy', 'max_depth': 3, 'min_..."
2,RandomizedSearchCV Decision Tree,0.7525,0.6948,"{'min_samples_split': 2, 'min_samples_leaf': 6..."
3,Random Forest,0.7655,0.7597,"n_estimators=100, random_state=42"


## Conclusion

In this project, a Decision Tree classifier was first evaluated using 5-fold cross-validation.

GridSearchCV and RandomizedSearchCV were then used to tune the Decision Tree hyperparameters. Both tuning methods improved the cross-validation accuracy compared with the baseline model.

Finally, a Random Forest ensemble model was trained and evaluated. Random Forest achieved the best overall performance with a CV accuracy of 76.55% and a test accuracy of 75.97%.

Therefore, Random Forest was selected as the best-performing model for this dataset.